In [1]:
# 1. Install necessary libraries
!pip install kagglehub textblob wordcloud

import pandas as pd
import numpy as np
import os
import string
import nltk
import pickle
import kagglehub
from nltk.corpus import stopwords
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import PassiveAggressiveClassifier
from sklearn.metrics import accuracy_score, confusion_matrix
from textblob import TextBlob
import matplotlib.pyplot as plt
from wordcloud import WordCloud

nltk.download('stopwords')
stop_words = set(stopwords.words('english'))

print("Libraries Imported Successfully! ✅")

Libraries Imported Successfully! ✅


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


In [2]:
# 2. Download dataset
path = kagglehub.dataset_download("clmentbisaillon/fake-and-real-news-dataset")

# Load CSV files
fake_df = pd.read_csv(os.path.join(path, "Fake.csv"))
true_df = pd.read_csv(os.path.join(path, "True.csv"))

# Add Labels (0 for Fake, 1 for True)
fake_df['label'] = 0
true_df['label'] = 1

# Combine and Shuffle
df = pd.concat([fake_df, true_df], axis=0).reset_index(drop=True)
df = df.sample(frac=1).reset_index(drop=True)

# Keep only necessary columns to save memory
df = df[['text', 'label']]

print(f"Dataset Loaded: {len(df)} rows")

Using Colab cache for faster access to the 'fake-and-real-news-dataset' dataset.
Dataset Loaded: 44898 rows


In [3]:
def clean_text(text):
    text = text.lower()
    text = "".join([char for char in text if char not in string.punctuation]) # Remove punctuation
    text = " ".join([word for word in text.split() if word not in stop_words]) # Remove stopwords
    return text

# Apply cleaning (This may take 1-2 minutes)
print("Cleaning text... please wait.")
df['text'] = df['text'].apply(clean_text)
print("Text Cleaning Done! ✨")

Cleaning text... please wait.
Text Cleaning Done! ✨


In [4]:
# Split Data
x_train, x_test, y_train, y_test = train_test_split(df['text'], df['label'], test_size=0.2, random_state=42)

# TF-IDF Vectorization
tfidf_vectorizer = TfidfVectorizer(stop_words='english', max_df=0.7)
tfidf_train = tfidf_vectorizer.fit_transform(x_train)
tfidf_test = tfidf_vectorizer.transform(x_test)

# Passive Aggressive Classifier Training
pac = PassiveAggressiveClassifier(max_iter=50)
pac.fit(tfidf_train, y_train)

# Accuracy Check
y_pred = pac.predict(tfidf_test)
score = accuracy_score(y_test, y_pred)
print(f'🔥 Model Accuracy: {round(score*100, 2)}%')

🔥 Model Accuracy: 99.53%


In [5]:
def fact_trace_analyze(input_news):
    # 1. Prediction
    vec_input = tfidf_vectorizer.transform([input_news])
    prediction = pac.predict(vec_input)[0]
    result = "REAL" if prediction == 1 else "FAKE"

    # 2. Sentiment Analysis (Extra Feature)
    analysis = TextBlob(input_news)
    sentiment = "Emotional/Biased" if analysis.sentiment.polarity < 0 else "Neutral/Fact-based"

    # 3. Automated Mitigation Message (Extra Feature)
    mitigation_msg = f"ALERT: The info you shared is flagged as {result}. Please refer to official sources like WHO/Reuters. Sentiment: {sentiment}"

    print(f"--- ANALYSIS REPORT ---")
    print(f"Prediction: {result}")
    print(f"Tone: {sentiment}")
    print(f"Correction Message to Spreader: {mitigation_msg}")
    print("------------------------")

# Test it!
sample_news = "The government is giving free iPhones to everyone because of the new law."
fact_trace_analyze(sample_news)

--- ANALYSIS REPORT ---
Prediction: FAKE
Tone: Neutral/Fact-based
Correction Message to Spreader: ALERT: The info you shared is flagged as FAKE. Please refer to official sources like WHO/Reuters. Sentiment: Neutral/Fact-based
------------------------


In [7]:
# Save the model and vectorizer
pickle.dump(pac, open('facttrace_model.pkl', 'wb'))
pickle.dump(tfidf_vectorizer, open('tfidf_vectorizer.pkl', 'wb'))

print("✅ Files saved: 'facttrace_model.pkl' and 'tfidf_vectorizer.pkl'")


✅ Files saved: 'facttrace_model.pkl' and 'tfidf_vectorizer.pkl'
